In [1]:
import os
import platform
import sys

import numpy as np
import h5py
import pint
import tables

In [2]:
match platform.system():
    case "Linux":
        sys.path.insert(1, os.path.abspath(".."))
        import lysis
        from lysis.util import Q_
        lysis_root = os.path.join("/", "home", "bpaynter", "git", "UCO-OpResearch", "lysis")
    case "Windows":
        import src.python.lysis as lysis

In [3]:
#Need to make considerations for loading the data from an existing HDF5 instead of just loading a new one every time.
#Need to start moving this notebook to lysis\src\python\lysis\util\ and replace datastore.py with this code becoming methods.
#Need to create some sort of class
r = lysis.util.Run(os.path.join(lysis_root, "data"), run_code="2024-04-16-1923")
r.read_file()
r.macro_params.total_molecules

21105

In [4]:
r.to_file()

Creates a variable f that references the file we want to access.
Need to change variables according to the name of you .h5 file

In [ ]:
f = h5py.File('schema_view.h5', 'w')

Creates folder and data set structure

In [ ]:
runs = f.create_group("1-PKd") 
micro_data = runs.create_group("micro_data") 
macro_data = runs.create_group("macro_data")


#Micro data set initializations
pli_first_time = micro_data.create_dataset("pli_first_time", (r.micro_params.simulations, ), dtype= np.float64)
tpa_final_num = micro_data.create_dataset("tpa_final_num", (r.micro_params.simulations, ), dtype= np.uint8)
fiber_degraded = micro_data.create_dataset("fiber_degraded", (r.micro_params.simulations, ), dtype= np.bool_)
sim_final_time = micro_data.create_dataset("sim_final_time", (r.micro_params.simulations, ), dtype= np.float64)
pli_generated_num = micro_data.create_dataset("pli_generated_num", (r.micro_params.simulations, ), dtype= np.uint16)
tpa_leaving_time = micro_data.create_dataset("tpa_leaving_time", (r.micro_params.simulations, ), dtype= np.float64)
tpa_unbound_by_pli = micro_data.create_dataset("tpa_unbound_by_pli", (r.micro_params.simulations, ), dtype= np.bool_)
tpa_unbound_kinetic = micro_data.create_dataset("tpa_unbound_kinetic", (r.micro_params.simulations, ), dtype = np.bool_)

#Macro data set initializations

for i in range(0 , r.macro_params.simulations):
    simulation = macro_data.create_group(f"sim_{i:02}")
    fiber_degrade_time = simulation.create_dataset("fiber_degrade_time", (1 , 3) , dtype=np.float64, maxshape = (None, 3))
    tpa_bind_events = simulation.create_dataset("tpa_bind_events", (1 , 10) , dtype=np.float64 , maxshape = (None, 10)) #snapshot time = 
    snapshot_time = simulation.create_dataset("snapshot_time", (1 , 1) , dtype=np.float64 , maxshape = (None, 1))
    #tpa_location_snapshot = simulation.create_dataset("tpa_location_snapshot", (r.macro_params.total_molecules , 1) , dtype=np.int32 , maxshape = (r.macro_params.total_molecules, None)) #I uncapped the max number of rows so the data can fit
    tpa_location_snapshot = simulation.create_dataset("tpa_location_snapshot", (r.macro_params.total_molecules , 1) , dtype=np.int32 , maxshape = (None, None))
    tpa_transit_time = simulation.create_dataset("tpa_transit_time", (r.macro_params.total_molecules , 1) , dtype=np.float64)
    


Reads in Micro scale data into file system

In [ ]:
file_code = "PLG2_tPA01_TB-xiii"

pli_first_time[:] = np.fromfile(
    os.path.join(r.os_path, f"firstPLi_{file_code}.dat"),
)

tpa_final_num[:] = np.fromfile(os.path.join(r.os_path, f"lasttPA_{file_code}.dat"), dtype=np.int32)

fiber_degraded[:] = np.fromfile(
    os.path.join(r.os_path, f"lyscomplete_{file_code}.dat"), 
    dtype=np.int32
).astype(bool)

sim_final_time[:] = np.fromfile(os.path.join(r.os_path, f"lysis_{file_code}.dat"))

pli_generated_num[:] = np.fromfile(os.path.join(r.os_path, f"PLi_{file_code}.dat"), dtype=np.int32).astype(np.uint16)

tpa_leaving_time[:] = np.fromfile(os.path.join(r.os_path, f"tPA_time_{file_code}.dat"))

tpa_unbound_by_pli[:] = np.fromfile(
    os.path.join(r.os_path, f"tPAPLiunbd_{file_code}.dat"), 
    dtype=np.int32
).astype(bool)

tpa_unbound_kinetic[:] = np.fromfile(
    os.path.join(r.os_path, f"tPAPLiunbd_{file_code}.dat"), 
    dtype=np.int32
).astype(bool)

Reads in Macro scale Fiber Degrade Time Data

In [ ]:

for i in range (0 , 10):
    file_reference = f[f"1-PKd/macro_data/sim_0{i}/fiber_degrade_time"]
    macro_file_code = f"TB-xiii__21_105_0{i}"
    data = np.loadtxt(os.path.join(r.os_path, f"0{i}\\f_deg_list_{macro_file_code}.dat") , delimiter = ",")
    data = np.reshape(data, (-1, 3))
    data[: , 1] = data[: , 1] - 1   #The data in column 1 is 1 indexed, so we need to convert it to 0 indexed
    file_reference.resize(data.shape)
    file_reference[:] = data
    #Adding Attributes to datasets
    file_reference.attrs["units"] = ["second" , "dimensionless" , "second"]

Reads in Macro scale TPA Bind Events Data | Needs to be reworked for efficiency 51 seconds

I believe what is taking so long is the resizing of the HDF dataset that is going on. Perhaps by specifying size on instatiation, we can avoid this.

In [ ]:
for i in range (0 , 10):
    file_reference = f[f"1-PKd/macro_data/sim_0{i}/tpa_bind_events"]
    macro_file_code = f"TB-xiii__21_105_0{i}"
    data = np.loadtxt(os.path.join(r.os_path, f"0{i}\\m_bind_t_{macro_file_code}.dat") , delimiter = ",")
    data = np.reshape(data, (-1, 4))
    data[:,[1,3]] = data[:,[1,3]] - 1   #The data in column 1 and 3 is 1 indexed, so we need to convert it to 0 indexed
    file_reference.resize(data.shape)
    file_reference[:] = data
    

Reads in Macro scale snapshot time data

In [ ]:
for i in range (0 , 10):
    file_reference = f[f"1-PKd/macro_data/sim_0{i}/tpa_location_snapshot"]
    macro_file_code = f"TB-xiii__21_105_0{i}"
    data = np.fromfile(os.path.join(r.os_path, f"0{i}\\m_loc_{macro_file_code}.dat") , dtype = np.int32).reshape(-1, r.macro_params.total_molecules)
    data = data - 1 #The data is 1 indexed, so we need to convert it to 0 indexed
    file_reference.resize(data.shape)
    file_reference[:] = data

Reads in Macro scale TPA Transit Time Data

In [ ]:
for i in range (0 , 10):
    file_reference = f[f"1-PKd/macro_data/sim_0{i}/tpa_transit_time"]
    macro_file_code = f"TB-xiii__21_105_0{i}"
    data = np.fromfile(os.path.join(r.os_path, f"0{i}\\mfpt_{macro_file_code}.dat") , dtype = np.float64).reshape(-1,1)
    file_reference[:] = data

Reads in Macro scale Snapshot Time Data

In [ ]:
for i in range (0 , 10):
    file_reference = f[f"1-PKd/macro_data/sim_0{i}/snapshot_time"]
    macro_file_code = f"TB-xiii__21_105_0{i}"
    data = np.fromfile(os.path.join(r.os_path, f"0{i}\\tsave_{macro_file_code}.dat") , dtype = np.float64).reshape(-1,1)
    file_reference.resize(data.shape)
    file_reference[:] = data

In [ ]:
f.close()

Post Processing

In [ ]:
import pint
u = pint.UnitRegistry()
Q = u.Quantity

In [ ]:
H5_dataset = f["1-PKd/macro_data/sim_00/fiber_degrade_time"]
unit_array = np.array(H5_dataset.attrs["units"])
dataset = np.array(f["1-PKd/macro_data/sim_00/fiber_degrade_time"])

legs2 = [400.0, 300.0] * u.centimeter
legs2 = dataset * u.dimensionless
print(legs2)
#print(legs2)

